# Agent: simulate

Develop and test **`agentic_scd.agents.simulate.simulate_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    UP1["classifications"]:::faded --> E1
    UP2["impacts"]:::faded --> E1
    subgraph E["simulate_node (batch aggregate)"]
        E1["risk = aggregate_risk()<br/>affected = sum(entities)"] --> E2["stockout_p = clamp(risk * (0.8 + 0.04*affected))"]
        E1 --> E3["revenue_impact = risk * REVENUE * (1 + 0.1*affected)"]
    end
    E2 --> D["simulation<br/>Simulation(stockout_probability, revenue_impact, assumptions)"]
    E3 --> D
    D --> DOWN["downstream: recommend"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `classifications` and `impacts`
- **Writes:** `simulation` (`Simulation`: stockout_probability, revenue_impact, assumptions)
- **Fallback / degradation:** no risk / no impacts → near-zero stockout probability and revenue impact

**Phase 6** replaces this with a SimPy discrete-event model + Monte Carlo runs, behind the same `simulate_node` signature.

## Is the DB up? (optional)

In [ ]:
# Optional: this agent runs fine offline on synthetic sample state. This snippet just
# reports whether the live DB is reachable (Setup section of 00_orchestration brings
# it up).
from agentic_scd.devtools import db_status

status = db_status()
print(status.detail)
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")

## Build a representative input state

In [ ]:
from agentic_scd.agents.classify import classify_node
from agentic_scd.agents.impact import impact_node
from agentic_scd.devtools import sample_state

# simulate reads BOTH classifications and impacts.
state = sample_state(count=2)
state.update(classify_node(state))
state.update(impact_node(state))
print("affected entities:", sum(len(i.affected_entities) for i in state["impacts"]))

## Call `simulate_node` in isolation

In [ ]:
from agentic_scd.agents.simulate import simulate_node

state.update(simulate_node(state))
sim = state["simulation"]
print(f"stockout probability: {sim.stockout_probability:.0%}")
print(f"revenue impact:       {sim.revenue_impact:,.0f}")
print(f"assumptions:          {sim.assumptions}")

## Iterate here

This is your dev surface: tweak the input above, re-run, and watch `simulate_node`'s output change. When you deepen this agent in its phase, keep the node signature the same so the rest of the graph is unaffected.